<a href="https://colab.research.google.com/github/netajik/FVL-class-activity/blob/master/pytorch_ngram_next_word_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Next-Word Prediction with an N-Gram Language Model (PyTorch)

**Problem:** Starting with a small text corpus, tokenize sentences, construct a vocabulary, form N-gram input/target examples, train a neural language model using PyTorch, and predict the next word from the preceding `N - 1` words.

**Learning goals:** Understand each stage of the pipeline and test it with reproducible examples. This notebook runs on **Google Colab (CPU is sufficient)** without downloading data. We use a **trigram (`N = 3`)** by default; change `N_GRAM` to experiment with other sizes.

> This is a small demonstration corpus, not a production language model. Predictions depend on the training sentences and vocabulary.

In [1]:
# 1. Imports and reproducibility
import random
import re
from collections import Counter

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

PyTorch version: 2.11.0+cu130
Device: cuda


In [2]:
# 2. Create a small, editable corpus
# Multiple sentences share contexts so the model must learn next-word distributions.
corpus = [
    "I love machine learning",
    "I love deep learning",
    "I love natural language processing",
    "I enjoy machine learning",
    "We study machine learning",
    "We study deep learning",
    "Machine learning is interesting",
    "Deep learning is powerful",
    "Natural language processing is interesting",
    "Machine learning uses data",
    "Deep learning uses neural networks",
    "Neural networks learn patterns",
    "Language models predict words",
    "Language models learn patterns",
    "We build language models",
    "I enjoy learning new concepts",
]
print("Number of sentences:", len(corpus))

Number of sentences: 16


In [3]:
# 3. Normalize and tokenize
# This simple tokenizer keeps words and contractions; it drops other punctuation.
# More sophisticated tokenization would be useful for real-world datasets.
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

tokenized_sentences = [tokenize(sentence) for sentence in corpus]
assert all(tokenized_sentences), "Every sentence should contain at least one token."
print("Original:", corpus[0])
print("Tokens:  ", tokenized_sentences[0])

Original: I love machine learning
Tokens:   ['i', 'love', 'machine', 'learning']


In [4]:
# 4. Build a vocabulary with explicit special tokens
PAD_TOKEN = "<pad>"  # Context padding at the beginning of sentences
UNK_TOKEN = "<unk>"  # Unseen word at inference time
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN]


def build_vocab(sentences: list[list[str]]) -> tuple[dict[str, int], dict[int, str]]:
    counts = Counter(word for sentence in sentences for word in sentence)
    # Sorting makes token IDs reproducible across runs.
    vocab_words = sorted(counts)
    word_to_id = {word: idx for idx, word in enumerate(SPECIAL_TOKENS + vocab_words)}
    id_to_word = {idx: word for word, idx in word_to_id.items()}
    return word_to_id, id_to_word


word_to_id, id_to_word = build_vocab(tokenized_sentences)
PAD_ID = word_to_id[PAD_TOKEN]
UNK_ID = word_to_id[UNK_TOKEN]
print("Vocabulary size:", len(word_to_id))
print("First vocabulary entries:", list(word_to_id.items())[:12])

Vocabulary size: 28
First vocabulary entries: [('<pad>', 0), ('<unk>', 1), ('build', 2), ('concepts', 3), ('data', 4), ('deep', 5), ('enjoy', 6), ('i', 7), ('interesting', 8), ('is', 9), ('language', 10), ('learn', 11)]


In [5]:
# 5. Generate N-gram (context, target) pairs
# A trigram uses two previous words to predict the next one.
# Left padding allows the model to learn predictions at sentence beginnings.
N_GRAM = 3
assert N_GRAM >= 2, "N_GRAM must be at least 2."
CONTEXT_SIZE = N_GRAM - 1


def generate_ngram_examples(sentences: list[list[str]], context_size: int,
                            vocab: dict[str, int]) -> list[tuple[list[int], int]]:
    examples = []
    for sentence in sentences:
        encoded = [vocab.get(word, vocab[UNK_TOKEN]) for word in sentence]
        padded = [vocab[PAD_TOKEN]] * context_size + encoded
        for target_pos in range(context_size, len(padded)):
            context = padded[target_pos - context_size:target_pos]
            target = padded[target_pos]
            examples.append((context, target))
    return examples


examples = generate_ngram_examples(tokenized_sentences, CONTEXT_SIZE, word_to_id)
print("Total training examples:", len(examples))
for context, target in examples[:7]:
    print([id_to_word[idx] for idx in context], "->", id_to_word[target])

Total training examples: 68
['<pad>', '<pad>'] -> i
['<pad>', 'i'] -> love
['i', 'love'] -> machine
['love', 'machine'] -> learning
['<pad>', '<pad>'] -> i
['<pad>', 'i'] -> love
['i', 'love'] -> deep


In [6]:
# 6. Dataset and DataLoader
class NGramDataset(Dataset):
    def __init__(self, pairs: list[tuple[list[int], int]]):
        self.pairs = pairs

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        context, target = self.pairs[index]
        return (torch.tensor(context, dtype=torch.long),
                torch.tensor(target, dtype=torch.long))


dataset = NGramDataset(examples)
loader = DataLoader(dataset, batch_size=16, shuffle=True)
example_inputs, example_targets = next(iter(loader))
assert example_inputs.shape[1] == CONTEXT_SIZE
assert example_inputs.dtype == torch.long and example_targets.dtype == torch.long
print("Input batch shape:", tuple(example_inputs.shape))
print("Target batch shape:", tuple(example_targets.shape))

Input batch shape: (16, 2)
Target batch shape: (16,)


In [7]:
# 7. Neural N-gram model: embeddings -> hidden layer -> vocabulary logits
# CrossEntropyLoss expects raw logits, so there is NO softmax in forward().
class NeuralNGramModel(nn.Module):
    def __init__(self, vocab_size: int, context_size: int,
                 embedding_dim: int = 32, hidden_dim: int = 64):
        super().__init__()
        self.context_size = context_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, context_ids: torch.Tensor) -> torch.Tensor:
        if context_ids.ndim != 2 or context_ids.shape[1] != self.context_size:
            raise ValueError(f"Expected input shape [batch, {self.context_size}]")
        embedded = self.embedding(context_ids)
        flattened = embedded.reshape(context_ids.shape[0], -1)
        return self.output(self.activation(self.hidden(flattened)))


model = NeuralNGramModel(len(word_to_id), CONTEXT_SIZE).to(DEVICE)
with torch.no_grad():
    logits = model(example_inputs.to(DEVICE))
assert logits.shape == (len(example_inputs), len(word_to_id))
print("Model output shape:", tuple(logits.shape))

Model output shape: (16, 28)


In [8]:
# 8. Train the model
# In this educational exercise we fit the entire small corpus; reported
# accuracy is TRAINING accuracy, not generalization to unseen text.
def train_model(model: nn.Module, data_loader: DataLoader,
                epochs: int = 180, lr: float = 0.01) -> list[float]:
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        cumulative_loss = 0.0
        total = 0
        for contexts, targets in data_loader:
            contexts, targets = contexts.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            predictions = model(contexts)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
            cumulative_loss += loss.item() * targets.size(0)
            total += targets.size(0)
        epoch_loss = cumulative_loss / total
        history.append(epoch_loss)
        if epoch == 1 or epoch % 30 == 0 or epoch == epochs:
            print(f"Epoch {epoch:3d}/{epochs} | training loss: {epoch_loss:.4f}")
    return history


loss_history = train_model(model, loader)

Epoch   1/180 | training loss: 3.1758
Epoch  30/180 | training loss: 0.7454
Epoch  60/180 | training loss: 0.7190
Epoch  90/180 | training loss: 0.7418
Epoch 120/180 | training loss: 0.7128
Epoch 150/180 | training loss: 0.7161
Epoch 180/180 | training loss: 0.7110


In [9]:
# 9. Inspect training performance
@torch.no_grad()
def training_accuracy(model: nn.Module, data_loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    for contexts, targets in data_loader:
        logits = model(contexts.to(DEVICE))
        predicted = logits.argmax(dim=1).cpu()
        correct += (predicted == targets).sum().item()
        total += targets.numel()
    return correct / total


accuracy = training_accuracy(model, DataLoader(dataset, batch_size=32))
print(f"Training next-word accuracy: {accuracy:.1%}")
print(f"Initial training loss: {loss_history[0]:.4f}")
print(f"Final training loss:   {loss_history[-1]:.4f}")
assert all(torch.isfinite(torch.tensor(loss_history)))
assert loss_history[-1] < loss_history[0], "Expected the training loss to decrease."
# Note: Repeated contexts can have different correct next words, so even a
# well-fitted model cannot achieve 100% top-1 accuracy on such examples.

Training next-word accuracy: 66.2%
Initial training loss: 3.1758
Final training loss:   0.7110


In [10]:
# 10. Predict the next word and show top-k probabilities
@torch.no_grad()
def predict_next_words(prompt: str, model: nn.Module, vocab: dict[str, int],
                       reverse_vocab: dict[int, str], context_size: int,
                       top_k: int = 3) -> list[tuple[str, float]]:
    if top_k < 1:
        raise ValueError("top_k must be at least 1")
    tokens = tokenize(prompt)
    # The model only needs the final N-1 words of a longer prompt.
    tokens = tokens[-context_size:]
    tokens = [PAD_TOKEN] * (context_size - len(tokens)) + tokens
    token_ids = [vocab.get(word, vocab[UNK_TOKEN]) for word in tokens]
    inputs = torch.tensor([token_ids], dtype=torch.long, device=DEVICE)

    model.eval()
    logits = model(inputs)[0].clone()
    # Never output the artificial padding and unknown tokens as a prediction.
    logits[vocab[PAD_TOKEN]] = float("-inf")
    logits[vocab[UNK_TOKEN]] = float("-inf")
    probabilities = torch.softmax(logits, dim=-1)
    available_words = len(vocab) - len(SPECIAL_TOKENS)
    values, indices = torch.topk(probabilities, k=min(top_k, available_words))
    return [(reverse_vocab[idx.item()], round(prob.item(), 4))
            for prob, idx in zip(values, indices)]


def predict_next_word(prompt: str) -> str:
    """Convenient interface that returns the highest-probability next word."""
    return predict_next_words(prompt, model, word_to_id, id_to_word,
                              CONTEXT_SIZE, top_k=1)[0][0]


for text in ["love machine", "machine learning", "deep learning", "I love"]:
    print(f"{text!r} -> {predict_next_words(text, model, word_to_id, id_to_word, CONTEXT_SIZE)}")

'love machine' -> [('learning', 1.0), ('machine', 0.0), ('language', 0.0)]
'machine learning' -> [('is', 0.6603), ('uses', 0.3382), ('new', 0.0004)]
'deep learning' -> [('is', 0.6406), ('uses', 0.3576), ('machine', 0.0003)]
'I love' -> [('deep', 0.4329), ('machine', 0.3478), ('natural', 0.2121)]


In [11]:
# 11. Edge-case tests and simple text generation
# Unknown words are mapped to <unk>; empty prompts are padded.
assert len(predict_next_words("", model, word_to_id, id_to_word, CONTEXT_SIZE)) == 3
assert isinstance(predict_next_word("never-seen vocabulary"), str)
assert isinstance(predict_next_word("I really love machine learning"), str)


def generate_text(prompt: str, steps: int = 5) -> str:
    """Generate a fixed number of words using greedy next-word prediction."""
    if steps < 0:
        raise ValueError("steps cannot be negative")
    words = tokenize(prompt)
    for _ in range(steps):
        next_word = predict_next_word(" ".join(words))
        words.append(next_word)
    return " ".join(words)


print("Generated example:", generate_text("I love", steps=5))
print("Validation checks passed.")

Generated example: i love deep learning is interesting deep
Validation checks passed.


## Notes and limitations

- `N_GRAM = 3` means each prediction uses the **previous two** words. Try `N_GRAM = 2` (bigram) or `4` (four-gram), then rerun all cells.
- We use **sentence-start padding** so the initial words also become targets. There is no end-of-sentence token, so the demonstration generates a fixed number of words rather than learning when to stop.
- The corpus contains **ambiguous contexts** (e.g., `I love` can be followed by several words). The model outputs a probability distribution rather than a guaranteed unique answer.
- The displayed accuracy is **training accuracy only**. For meaningful generalization measurement, collect a larger corpus and split by sentence into training, validation, and test sets before building the vocabulary and fitting the model.
- Unknown input words map to `<unk>`; this small corpus cannot reliably predict from unfamiliar phrases.